# 利己的な遺伝子シミュレーション — 分析ノートブック

論文第5章（Results & Discussion）用のグラフを生成します。

## 目次
1. セットアップ
2. 図1：ベースケースの協力率時系列推移
3. 図2：シナリオA — リソース量別の最終協力率比較
4. 図3：シナリオB — 認識誤り率εと協力率の関係
5. 図4：シナリオC — 移動性と協力率の関係
6. 図5：シナリオCの空間スナップショット比較
7. サマリー統計

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

from model import (
    SelfishGeneModel,
    DEFAULT_PARAMS,
    SCENARIO_A_RICH, SCENARIO_A_POOR,
    SCENARIO_B_PERFECT, SCENARIO_B_ERROR05, SCENARIO_B_ERROR10, SCENARIO_B_ERROR20, SCENARIO_B_NONE,
    SCENARIO_C_FIXED, SCENARIO_C_LOW, SCENARIO_C_MID, SCENARIO_C_HIGH,
)

# 出力先の設定
FIGURES_DIR = os.path.join(os.getcwd(), '..', 'data', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# グラフスタイルの設定（論文品質）
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

N_STEPS = 500  # run_analysis.py と統一（収束確認に十分な 500 ステップ）
print(f'設定完了。グラフ出力先: {FIGURES_DIR}')

In [ ]:
def run_simulation(params: dict, n_steps: int = N_STEPS) -> pd.DataFrame:
    """指定パラメータでシミュレーションを実行し、DataFrameを返す。"""
    model = SelfishGeneModel(**params)
    for _ in range(n_steps):
        model.step()
        if len(model.agents) == 0:
            break  # 個体群絶滅時は早期終了
    return model.datacollector.get_model_vars_dataframe()

print('run_simulation 関数を定義しました。')

---
## 図1：ベースケースの協力率時系列推移

In [ ]:
print('ベースケースを実行中...')
df_base = run_simulation(DEFAULT_PARAMS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：協力率の推移
ax = axes[0]
ax.plot(df_base['CoopRatio'], color='#3498db', linewidth=1.8, label='協力率')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.0, alpha=0.6, label='50%ライン')
ax.set_xlabel('ステップ')
ax.set_ylabel('協力率')
ax.set_ylim(0, 1)
ax.set_title('(a) 協力率の時系列推移')
ax.legend()
ax.grid(True, alpha=0.3)

# 右：個体数と戦略別内訳
ax2 = axes[1]
ax2.stackplot(
    df_base.index,
    df_base['Cooperators'], df_base['Defectors'],
    labels=['協力者', '裏切り者'],
    colors=['#3498db', '#e74c3c'],
    alpha=0.7
)
ax2.set_xlabel('ステップ')
ax2.set_ylabel('個体数')
ax2.set_title('(b) 戦略別個体数の推移')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

fig.suptitle('図1：ベースケース (diffusion_rate=0.1, mutation_rate=0.01)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig1_baseline.png'))
plt.show()
print(f'最終協力率: {df_base["CoopRatio"].iloc[-1]:.3f}')

---
## 図2：シナリオA — リソース量別の最終協力率比較

In [ ]:
resource_rates = [0.2, 0.5, 1.0, 2.0, 3.0, 5.0]
results_a = {}

for rr in resource_rates:
    print(f'  resource_recovery_rate={rr} を実行中...')
    params = {**DEFAULT_PARAMS, 'resource_recovery_rate': rr}
    df = run_simulation(params)
    results_a[rr] = df

print('シナリオA完了')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors_a = plt.cm.Blues(np.linspace(0.3, 0.9, len(resource_rates)))

# 左：各条件の時系列
ax = axes[0]
for i, rr in enumerate(resource_rates):
    ax.plot(results_a[rr]['CoopRatio'], color=colors_a[i],
            linewidth=1.5, label=f'rate={rr}')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlabel('ステップ')
ax.set_ylabel('協力率')
ax.set_ylim(0, 1)
ax.set_title('(a) リソース量別の協力率推移')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 右：最終協力率の棒グラフ
ax2 = axes[1]
final_coops = [results_a[rr]['CoopRatio'].iloc[-1] for rr in resource_rates]
bars = ax2.bar([str(rr) for rr in resource_rates], final_coops,
               color=colors_a, edgecolor='black', linewidth=0.5)
ax2.axhline(0.5, color='red', linestyle='--', linewidth=1.0, alpha=0.7, label='50%ライン')
ax2.set_xlabel('resource_recovery_rate')
ax2.set_ylabel('最終協力率')
ax2.set_ylim(0, 1)
ax2.set_title('(b) リソース量別の最終協力率')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, final_coops):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f'{val:.2f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('図2：シナリオA — リソース量と協力率の関係', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig2_scenario_a.png'))
plt.show()

---
## 図3：シナリオB — 認識誤り率εと協力率の関係（緑ひげ効果）

In [ ]:
scenario_b_configs = {
    'ε=0.0 (完全認識)': SCENARIO_B_PERFECT,
    'ε=0.05':           SCENARIO_B_ERROR05,
    'ε=0.10':           SCENARIO_B_ERROR10,
    'ε=0.20':           SCENARIO_B_ERROR20,
    '緑ひげなし':        SCENARIO_B_NONE,
}

results_b = {}
for label, params in scenario_b_configs.items():
    print(f'  {label} を実行中...')
    results_b[label] = run_simulation(params)

print('シナリオB完了')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors_b = ['#1a237e', '#3949ab', '#7986cb', '#c5cae9', '#e53935']

# 左：時系列推移
ax = axes[0]
for (label, df), color in zip(results_b.items(), colors_b):
    ax.plot(df['CoopRatio'], color=color, linewidth=1.5,
            linestyle='--' if label == '緑ひげなし' else '-',
            label=label)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
ax.set_xlabel('ステップ')
ax.set_ylabel('協力率')
ax.set_ylim(0, 1)
ax.set_title('(a) 認識誤り率別の協力率推移')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 右：最終協力率の比較
ax2 = axes[1]
labels_b = list(results_b.keys())
finals_b = [results_b[l]['CoopRatio'].iloc[-1] for l in labels_b]
bars = ax2.barh(labels_b, finals_b, color=colors_b, edgecolor='black', linewidth=0.5)
ax2.axvline(0.5, color='red', linestyle='--', linewidth=1.0, alpha=0.7)
ax2.set_xlabel('最終協力率')
ax2.set_xlim(0, 1)
ax2.set_title('(b) 条件別の最終協力率')
ax2.grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, finals_b):
    ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=9)

fig.suptitle('図3：シナリオB — 緑ひげ効果と認識誤り率εの影響', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig3_scenario_b.png'))
plt.show()

---
## 図4：シナリオC — 移動性と協力率の関係

In [ ]:
scenario_c_configs = {
    '固定 (d=0.0)':    SCENARIO_C_FIXED,
    '低移動 (d=0.1)':  SCENARIO_C_LOW,
    '中移動 (d=0.3)':  SCENARIO_C_MID,
    '高移動 (d=0.9)':  SCENARIO_C_HIGH,
}

results_c = {}
for label, params in scenario_c_configs.items():
    print(f'  {label} を実行中...')
    results_c[label] = run_simulation(params)

print('シナリオC完了')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_c = ['#1b5e20', '#388e3c', '#81c784', '#c8e6c9']

# 左：時系列推移
ax = axes[0]
for (label, df), color in zip(results_c.items(), colors_c):
    ax.plot(df['CoopRatio'], color=color, linewidth=1.8, label=label)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlabel('ステップ')
ax.set_ylabel('協力率')
ax.set_ylim(0, 1)
ax.set_title('(a) 移動性別の協力率推移')
ax.legend()
ax.grid(True, alpha=0.3)

# 右：移動性 vs 最終協力率（折れ線）
ax2 = axes[1]
d_values = [0.0, 0.1, 0.3, 0.9]
finals_c = [results_c[l]['CoopRatio'].iloc[-1] for l in results_c.keys()]
ax2.plot(d_values, finals_c, 'o-', color='#2e7d32', linewidth=2, markersize=8)
ax2.axhline(0.5, color='red', linestyle='--', linewidth=1.0, alpha=0.7)
ax2.set_xlabel('diffusion_rate (移動性)')
ax2.set_ylabel('最終協力率')
ax2.set_ylim(0, 1)
ax2.set_title('(b) 移動性と最終協力率の関係')
ax2.grid(True, alpha=0.3)
for x, y in zip(d_values, finals_c):
    ax2.annotate(f'{y:.2f}', (x, y), textcoords='offset points',
                 xytext=(5, 5), fontsize=9)

fig.suptitle('図4：シナリオC — 移動性（diffusion_rate）と協力率の関係', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig4_scenario_c.png'))
plt.show()

---
## 図5：シナリオCの空間スナップショット比較

In [ ]:
snapshot_configs = [
    ('固定 (d=0.0)',   SCENARIO_C_FIXED),
    ('低移動 (d=0.1)', SCENARIO_C_LOW),
    ('高移動 (d=0.9)', SCENARIO_C_HIGH),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cmap = mcolors.ListedColormap(['#f0f0f0', '#3498db', '#e74c3c'])

for ax, (label, params) in zip(axes, snapshot_configs):
    print(f'  {label} のスナップショットを生成中...')
    model = SelfishGeneModel(**params)
    for _ in range(N_STEPS):
        model.step()
    snapshot = model.get_grid_snapshot()
    final_coop = model._coop_ratio()
    
    im = ax.imshow(snapshot.T, cmap=cmap, vmin=0, vmax=2, origin='lower')
    ax.set_title(f'{label}\n最終協力率: {final_coop:.1%}', fontsize=11)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

# 共通凡例
legend_elements = [
    Patch(facecolor='#3498db', label='協力者'),
    Patch(facecolor='#e74c3c', label='裏切り者'),
    Patch(facecolor='#f0f0f0', label='空'),
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=3, bbox_to_anchor=(0.5, -0.05), fontsize=11)

fig.suptitle(f'図5：シナリオC — {N_STEPS}ステップ後の空間分布', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig5_snapshots.png'), bbox_inches='tight')
plt.show()
print('スナップショット完了')

---
## サマリー統計（論文§5.1用）

In [ ]:
print('=' * 60)
print('サマリー統計')
print('=' * 60)

print('\n【ベースケース】')
print(f'  最終協力率: {df_base["CoopRatio"].iloc[-1]:.3f}')
print(f'  最終個体数: {df_base["TotalAgents"].iloc[-1]}')
print(f'  平均エネルギー: {df_base["MeanEnergy"].iloc[-1]:.2f}')

print('\n【シナリオA — 最終協力率】')
for rr in resource_rates:
    val = results_a[rr]['CoopRatio'].iloc[-1]
    print(f'  resource_rate={rr}: {val:.3f}')

print('\n【シナリオB — 最終協力率】')
for label, df in results_b.items():
    print(f'  {label}: {df["CoopRatio"].iloc[-1]:.3f}')

print('\n【シナリオC — 最終協力率】')
for label, df in results_c.items():
    print(f'  {label}: {df["CoopRatio"].iloc[-1]:.3f}')

# CSV保存
summary = pd.DataFrame({
    'scenario': ['baseline'] +
                [f'A_rate{rr}' for rr in resource_rates] +
                list(results_b.keys()) +
                list(results_c.keys()),
    'final_coop_ratio': [df_base['CoopRatio'].iloc[-1]] +
                        [results_a[rr]['CoopRatio'].iloc[-1] for rr in resource_rates] +
                        [df['CoopRatio'].iloc[-1] for df in results_b.values()] +
                        [df['CoopRatio'].iloc[-1] for df in results_c.values()],
})
csv_path = os.path.join(FIGURES_DIR, '..', 'summary_results.csv')
summary.to_csv(csv_path, index=False)
print(f'\nCSV保存: {csv_path}')
print('\n全グラフを', FIGURES_DIR, 'に保存しました。')